In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score

os.chdir("C:/Python/Datasets")

milk = pd.read_csv("milk.csv", index_col=0)
scaler = StandardScaler().set_output(transform='pandas')
df_scaled = scaler.fit_transform(milk)

clust_DB = DBSCAN(eps=1, min_samples=2)
clust_DB.fit(df_scaled)
print(clust_DB.labels_)

df_scaled['Clust'] = clust_DB.labels_
df_scl_inliers = df_scaled[df_scaled['Clust']!=-1]
print( silhouette_score(df_scl_inliers.iloc[:,:-1], df_scl_inliers.iloc[:,-1]) )

# Trying diferent value combinations

eps_range = np.linspace(0.01, 1.5, 10)
mp_range = [2,3,4,5]
cnt = 0
a =[]
for i in eps_range:
    for j in mp_range:
        clust_DB = DBSCAN(eps=i, min_samples=j)
        clust_DB.fit(df_scaled.iloc[:,:5])
        if len(set(clust_DB.labels_)) > 2:
            cnt = cnt + 1
            df_scaled['Clust'] = clust_DB.labels_
            df_scl_inliers = df_scaled[df_scaled['Clust']!=-1]
            sil_sc = silhouette_score(df_scl_inliers.iloc[:,:-1], df_scl_inliers.iloc[:,-1])
            a.append([cnt,i,j,sil_sc])

pa = pd.DataFrame(a,columns=['Sr','eps','min_pt','sil'])
pa.sort_values('sil', ascending=False)

# Clustering with best parameters

df_scaled = scaler.fit_transform(milk)

clust_DB = DBSCAN(eps=0.341111, min_samples=2)
clust_DB.fit(df_scaled)
clust_DB.labels_

df1 = milk.copy()
df1['Cluster'] = clust_DB.labels_
df1['Cluster'] = df1['Cluster'].astype(str)

sns.pairplot(data=df1, hue='Cluster')
plt.show()

# View w/o outliers

df1_in = df1[df1['Cluster']!="-1"]
sns.pairplot(data=df1_in, hue='Cluster')
plt.title("Only inliers plotted")
plt.show()